# Segmentação de Imagens Acústicas de Poço sem *Ground Truth* (WAID)

**ELE2346 – Deep Learning** · Mestrado em Engenharia Elétrica (Sistemas Inteligentes) · PUC-Rio · 2026.2
Autor: Jorge Alberto Avelino de Souza

**Notebook 05 – Ablações da Fase 3 (H4, H5, H3, H11 e H7).** Dados: WAID (PETROBRAS, CC BY 4.0), 5 poços.

## Fase 3 – Ablações
Cada ablação muda uma coisa na U-Net e é comparada com a referência do notebook 04 (U-Net, 3 sementes). Métrica principal: IoU sintético. Regra de leitura: há efeito se |média da diferença| > 1 desvio-padrão entre sementes e o sinal se repete em pelo menos 4 dos 5 poços.

### 5.1 Ambiente: bibliotecas, Google Drive e GPU
Importando as bibliotecas, montando o Drive e instalando a biblioteca de segmentação.

In [1]:
# Importando as bibliotecas usadas no notebook
import sys
from glob import glob
import pandas as pd
import torch
from google.colab import drive

In [2]:
# Montando o Drive para uso dos arquivos WAID
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# Instalando a biblioteca com a U-Net e o encoder ResNet18 (o Colab já traz o torch)
!pip install -q segmentation-models-pytorch

### 5.2 Configuração do projeto
Importando as funções de `src` e repetindo a configuração da referência (notebook 04), para que só a mudança da ablação difira.

In [4]:
# Adicionando ao caminho de busca do Python a pasta que contém "src"
sys.path.insert(0, "/content/drive/MyDrive/waid")

# Importando as constantes e as funções do projeto
from src.config import POCOS, NPY_DIR, CARAC_DIR, ANOT_DIR, RES_DIR, ALTURA_PATCH
from src.data.dataset_patches import preparar_dados, indice_sem_exclusao
from src.train.treino import CFG_PADRAO, rodar_lowo
from src.data.normalizacao import carregar_brutas, normalizar, limiares_otsu, parametros
from src.synth.comum import gerar_comum, projetar
from src.synth.sintetico import gerar_sintetico
from src.labels.limiares import limiar_multiotsu_poco
from src.eval.ablacao import tabela_ablacao
from src.eval.patches import avaliar_modelos, comparar_pareado
from src.eval.leitura_auc import ler_auc

# Escolhendo o dispositivo (GPU, se houver)
DISP = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("dispositivo:", DISP, torch.cuda.get_device_name(0) if DISP.type == "cuda" else "")

# Repetindo a configuração da referência (mesmas épocas, sementes e modelo)
CFG = dict(CFG_PADRAO, epocas=10)
MODELO = ["unet_resnet18"]
SEMENTES = [0, 1, 2]

dispositivo: cuda Tesla T4


### 5.3 Dados e referência
Carregando os dados e a tabela de resumos da referência (U-Net do notebook 04).

In [5]:
# Carregando os dados (imagens, limiar de Otsu por poço, patches sem sobreposição com os anotados)
imagens, limiares, indice_treino, anot = preparar_dados(NPY_DIR, CARAC_DIR, ANOT_DIR, POCOS, ALTURA_PATCH)

# Lendo os resumos da referência e ficando só com a U-Net
ref = pd.read_csv(f"{RES_DIR}/modelos_sint_resumos.csv")
ref = ref[ref.modelo == "unet_resnet18"]
print(len(ref), "folds de referência (esperado: 15)")

15 folds de referência (esperado: 15)


### 5.4 H4: sem deslocamento circular no azimute
A referência gira as colunas de cada patch por um valor aleatório no treino (aumento de dados). Aqui o giro fica desligado. Tempo previsto: cerca de 30 min.

In [6]:
# Treinando a U-Net sem o aumento por deslocamento circular (todo o resto igual à referência)
esc_h4, res_h4 = rodar_lowo(MODELO, SEMENTES, POCOS, imagens, indice_treino, anot, limiares,
                            dict(CFG, deslocar=False), DISP, f"{RES_DIR}/ablacoes/h4_sem_deslocamento")
res_h4.to_csv(f"{RES_DIR}/ablacoes/h4_resumos.csv", index=False)

### 5.5 H5: treinar sem excluir as zonas de artefato (D1)
O treino e a validação passam a incluir os patches das zonas excluídas pela D1. A avaliação (patches anotados e fundos do teste sintético) continua só nos patches válidos, e o limiar de Otsu de cada poço não muda. Tempo previsto: cerca de 30 min.

In [7]:
# Montando o índice de treino com as zonas da D1 incluídas (a avaliação usa o índice original)
indice_h5 = indice_sem_exclusao(imagens, anot, ALTURA_PATCH)
print(len(indice_treino), "patches no treino da referência;", len(indice_h5), "na H5")

# Treinando a U-Net com as zonas de artefato no treino e na validação
esc_h5, res_h5 = rodar_lowo(MODELO, SEMENTES, POCOS, imagens, indice_h5, anot, limiares,
                            CFG, DISP, f"{RES_DIR}/ablacoes/h5_sem_exclusao", indice_avaliacao=indice_treino)
res_h5.to_csv(f"{RES_DIR}/ablacoes/h5_resumos.csv", index=False)

3227 patches no treino da referência; 3274 na H5


### 5.6 H3: normalização global (em vez de por poço)
Os percentis 1 e 99 passam a ser calculados com os 5 poços juntos (sem as zonas da D1). O limiar de Otsu de cada poço é recalculado sobre a nova imagem. A avaliação usa os mesmos patches da referência. O Dice contra o pseudo-rótulo muda de significado nesta ablação; por isso a métrica principal é o IoU sintético. Tempo previsto: cerca de 30 min.

In [8]:
# Carregando a amplitude bruta (dB, com a D2 aplicada) e conferindo que a normalização por poço reproduz a referência
brutas, excl = carregar_brutas(NPY_DIR, POCOS)
ref_norm = normalizar(brutas, excl, "poco")
for p in POCOS:
    dif = abs(ref_norm[p].astype("float32") - imagens[p].astype("float32")).max()
    print(p, "diferença máxima para a referência:", float(dif))

antilope25 diferença máxima para a referência: 0.0
antilope37 diferença máxima para a referência: 0.0
botorosa47 diferença máxima para a referência: 0.0
coala88 diferença máxima para a referência: 0.0
tatu22 diferença máxima para a referência: 0.0


In [9]:
# Normalizando pelo conjunto dos poços e recalculando o limiar de Otsu de cada poço
imagens_g = normalizar(brutas, excl, "global")
limiares_g = limiares_otsu(imagens_g, excl)
print({p: round(v, 3) for p, v in limiares_g.items()})

# Treinando a U-Net com a normalização global (mesmos patches e mesma configuração da referência)
esc_h3g, res_h3g = rodar_lowo(MODELO, SEMENTES, POCOS, imagens_g, indice_treino, anot, limiares_g,
                              CFG, DISP, f"{RES_DIR}/ablacoes/h3_global")
res_h3g.to_csv(f"{RES_DIR}/ablacoes/h3_global_resumos.csv", index=False)

{'antilope25': 0.58, 'antilope37': 0.393, 'botorosa47': 0.682, 'coala88': 0.701, 'tatu22': 0.511}


### 5.7 H3: normalização por intervalo
Os percentis 1 e 99 passam a ser calculados em blocos de 2.000 linhas (cerca de 10 m) de cada poço. O tamanho do bloco é escolha do autor, sem apoio de especialista; os blocos com menos de 200 linhas válidas usam os percentis do poço. Tempo previsto: cerca de 30 min.

In [10]:
# Normalizando por bloco de linhas e recalculando o limiar de Otsu de cada poço
imagens_i = normalizar(brutas, excl, "intervalo")
limiares_i = limiares_otsu(imagens_i, excl)
print({p: round(v, 3) for p, v in limiares_i.items()})

# Treinando a U-Net com a normalização por intervalo
esc_h3i, res_h3i = rodar_lowo(MODELO, SEMENTES, POCOS, imagens_i, indice_treino, anot, limiares_i,
                              CFG, DISP, f"{RES_DIR}/ablacoes/h3_intervalo")
res_h3i.to_csv(f"{RES_DIR}/ablacoes/h3_intervalo_resumos.csv", index=False)

{'antilope25': 0.67, 'antilope37': 0.662, 'botorosa47': 0.666, 'coala88': 0.67, 'tatu22': 0.666}


### 5.8 Conferência do pseudo-rótulo
Em cada normalização, o limiar de Otsu muda e o pseudo-rótulo muda junto. Esta seção mede a fração de pixels marcados como escuros (abaixo do limiar) nas linhas válidas de cada poço. Serve para saber se o alvo do treino mudou de natureza, o que afeta a leitura do Dice contra o pseudo-rótulo.

In [11]:
# Calculando a fração de pixels abaixo do limiar de cada poço, em cada normalização (1 a cada 10 linhas válidas)
import numpy as np
linhas = []
for p in POCOS:
    valido = ~excl[p]
    linhas.append({"poco": p,
                   "frac_ref": float((imagens[p][valido][::10] < limiares[p]).mean()),
                   "frac_global": float((imagens_g[p][valido][::10] < limiares_g[p]).mean()),
                   "frac_intervalo": float((imagens_i[p][valido][::10] < limiares_i[p]).mean()),
                   "limiar_ref": limiares[p], "limiar_global": limiares_g[p], "limiar_intervalo": limiares_i[p]})
frac = pd.DataFrame(linhas).round(3)
print(frac.to_string(index=False))
frac.to_csv(f"{RES_DIR}/ablacoes/h3_fracao_escura.csv", index=False)

      poco  frac_ref  frac_global  frac_intervalo  limiar_ref  limiar_global  limiar_intervalo
antilope25     0.035        0.030           0.187       0.541          0.580             0.670
antilope37     0.106        0.157           0.150       0.623          0.393             0.662
botorosa47     0.155        0.129           0.164       0.670          0.682             0.666
   coala88     0.182        0.075           0.251       0.682          0.701             0.670
    tatu22     0.521        0.519           0.239       0.760          0.511             0.666


### 5.9 Leitura das ablações
Aplicando a regra de leitura à métrica principal (`iou_sint`) e às de apoio.


In [12]:
# Aplicando a regra de leitura a cada ablação (candidata menos referência, pareando modelo, semente e poço)
for nome, res in [("H4 sem deslocamento circular", res_h4), ("H5 sem exclusão da D1", res_h5),
                    ("H3 normalização global", res_h3g), ("H3 normalização por intervalo", res_h3i)]:
    print(nome)
    print(tabela_ablacao(ref, res).to_string(index=False))
    print()

H4 sem deslocamento circular
          metrica   ref  cand  delta  dp_sementes pocos_mesmo_sinal              veredito
         iou_sint 0.771 0.763 -0.008        0.043               3/5 sem efeito detectável
        dice_sint 0.824 0.819 -0.005        0.049               3/5 sem efeito detectável
      iou_sint_25 0.909 0.904 -0.005        0.034               2/5 sem efeito detectável
      iou_sint_75 0.887 0.852 -0.035        0.044               4/5 sem efeito detectável
      iou_sint_95 0.518 0.533  0.015        0.128               2/5 sem efeito detectável
dice_pseudo_teste 0.625 0.654  0.029        0.042               3/5 sem efeito detectável

H5 sem exclusão da D1
          metrica   ref  cand  delta  dp_sementes pocos_mesmo_sinal              veredito
         iou_sint 0.771 0.763 -0.008        0.042               3/5 sem efeito detectável
        dice_sint 0.824 0.819 -0.005        0.041               2/5 sem efeito detectável
      iou_sint_25 0.909 0.922  0.014        0.04

### 5.10 H3 com teste sintético comum
Na seção 5.9, a H3 foi avaliada em testes sintéticos diferentes, porque o limiar de Otsu (e com ele o contraste, os fundos e a região ignorada) muda com a normalização. Aqui o teste é gerado uma vez, na normalização da referência, e convertido para dB. Cada normalização é aplicada depois sobre as mesmas imagens, máscaras e regiões ignoradas. As duas H3 são treinadas de novo. Tempo previsto: cerca de 1 h.

In [13]:
# Gerando o teste sintético uma vez (referência), em dB, e projetando cada normalização sobre as mesmas imagens
params = {m: parametros(brutas, excl, m) for m in ["poco", "global", "intervalo"]}
comum = gerar_comum(imagens, brutas, params["poco"], indice_treino, limiares, POCOS, ALTURA_PATCH)
sint = {m: projetar(comum, params[m]) for m in params}

# Conferindo que a normalização por poço reproduz o teste sintético da referência (esperado: dif. máx. abaixo de 1e-3)
for p in POCOS:
    x0, m0, g0, _ = gerar_sintetico(imagens, indice_treino, limiares[p], p, altura=ALTURA_PATCH)
    xp, mp, gp, _ = sint["poco"][p]
    print(p, "dif. máx. de x:", round(float(abs(x0 - xp).max()), 5), "| máscaras iguais:", bool((m0 == mp).all()),
          "| regiões ignoradas iguais:", bool((g0 == gp).all()))

antilope25 dif. máx. de x: 0.00024 | máscaras iguais: True | regiões ignoradas iguais: True
antilope37 dif. máx. de x: 0.00024 | máscaras iguais: True | regiões ignoradas iguais: True
botorosa47 dif. máx. de x: 0.00024 | máscaras iguais: True | regiões ignoradas iguais: True
coala88 dif. máx. de x: 0.00024 | máscaras iguais: True | regiões ignoradas iguais: True
tatu22 dif. máx. de x: 0.00024 | máscaras iguais: True | regiões ignoradas iguais: True


In [14]:
# Treinando a U-Net com a normalização global, avaliada no teste sintético comum
esc_h3cg, res_h3cg = rodar_lowo(MODELO, SEMENTES, POCOS, imagens_g, indice_treino, anot, limiares_g,
                                CFG, DISP, f"{RES_DIR}/ablacoes/h3c_global", sint=sint["global"])
res_h3cg.to_csv(f"{RES_DIR}/ablacoes/h3c_global_resumos.csv", index=False)

In [15]:
# Treinando a U-Net com a normalização por intervalo, avaliada no teste sintético comum
esc_h3ci, res_h3ci = rodar_lowo(MODELO, SEMENTES, POCOS, imagens_i, indice_treino, anot, limiares_i,
                                CFG, DISP, f"{RES_DIR}/ablacoes/h3c_intervalo", sint=sint["intervalo"])
res_h3ci.to_csv(f"{RES_DIR}/ablacoes/h3c_intervalo_resumos.csv", index=False)

### 5.11 Leitura da H3 com teste sintético comum
Aplicando a regra de leitura só às métricas do teste sintético (o Dice contra o pseudo-rótulo não é comparável entre normalizações). A referência é a mesma da seção 5.9, pois o teste dela é idêntico ao teste comum (conferido na seção 5.10).

In [16]:
# Aplicando a regra de leitura às métricas sintéticas (candidata menos referência)
METRICAS_SINT = ["iou_sint", "dice_sint", "iou_sint_25", "iou_sint_75", "iou_sint_95"]
for nome, res in [("H3 normalização global (teste comum)", res_h3cg),
                  ("H3 normalização por intervalo (teste comum)", res_h3ci)]:
    print(nome)
    print(tabela_ablacao(ref, res, METRICAS_SINT).to_string(index=False))
    print()

H3 normalização global (teste comum)
    metrica   ref  cand  delta  dp_sementes pocos_mesmo_sinal              veredito
   iou_sint 0.771 0.521 -0.251        0.057               4/5                efeito
  dice_sint 0.824 0.575 -0.249        0.063               4/5                efeito
iou_sint_25 0.909 0.826 -0.082        0.039               3/5 sem efeito detectável
iou_sint_75 0.887 0.497 -0.390        0.113               4/5                efeito
iou_sint_95 0.518 0.239 -0.279        0.067               4/5                efeito

H3 normalização por intervalo (teste comum)
    metrica   ref  cand  delta  dp_sementes pocos_mesmo_sinal veredito
   iou_sint 0.771 0.366 -0.406        0.020               5/5   efeito
  dice_sint 0.824 0.467 -0.357        0.019               5/5   efeito
iou_sint_25 0.909 0.368 -0.541        0.033               5/5   efeito
iou_sint_75 0.887 0.369 -0.518        0.034               5/5   efeito
iou_sint_95 0.518 0.360 -0.158        0.055               4

### 5.12 H11: limiar do pseudo-rótulo (Multi-Otsu, classe mais escura)
O notebook 06 mostrou que, no tatu22 (e em trechos do botorosa47), o limiar de Otsu separa regimes de amplitude: patches inteiros ficam marcados como escuros. Aqui o pseudo-rótulo de treino usa o limiar que separa a classe mais escura do Multi-Otsu com 3 classes (3% a 6% de pixels escuros nos 5 poços). A imagem de entrada não muda. **Atenção à leitura:** o IoU do teste sintético comum define as feições pelo limiar de Otsu da referência; como a H11 muda a definição de "escuro", esse IoU não é neutro (ver README, item 77). A leitura principal da H11 é a AUC por patch no conjunto anotado (seção 5.15), que não usa o limiar. Tempo previsto: cerca de 30 min.

In [17]:
# Calculando o limiar do Multi-Otsu (classe mais escura) de cada poço
limiares_mo = {p: limiar_multiotsu_poco(imagens[p], excl[p]) for p in POCOS}
print({p: round(v, 3) for p, v in limiares_mo.items()})

# Treinando a U-Net com esse pseudo-rótulo e avaliando no teste sintético comum (normalização por poço)
esc_h11, res_h11 = rodar_lowo(MODELO, SEMENTES, POCOS, imagens, indice_treino, anot, limiares_mo,
                              CFG, DISP, f"{RES_DIR}/ablacoes/h11_multiotsu", sint=sint["poco"])
res_h11.to_csv(f"{RES_DIR}/ablacoes/h11_multiotsu_resumos.csv", index=False)

{'antilope25': 0.475, 'antilope37': 0.455, 'botorosa47': 0.49, 'coala88': 0.447, 'tatu22': 0.439}


### 5.13 Leitura da H11
Aplicando a regra de leitura às métricas do teste sintético. O Dice contra o pseudo-rótulo fica de fora, pois o pseudo-rótulo mudou.

In [18]:
# Aplicando a regra de leitura à H11 (candidata menos referência)
print("H11 pseudo-rótulo Multi-Otsu (teste comum)")
print(tabela_ablacao(ref, res_h11, METRICAS_SINT).to_string(index=False))

H11 pseudo-rótulo Multi-Otsu (teste comum)
    metrica   ref  cand  delta  dp_sementes pocos_mesmo_sinal veredito
   iou_sint 0.771 0.455 -0.317        0.024               5/5   efeito
  dice_sint 0.824 0.491 -0.333        0.031               5/5   efeito
iou_sint_25 0.909 0.999  0.091        0.030               5/5   efeito
iou_sint_75 0.887 0.357 -0.530        0.021               5/5   efeito
iou_sint_95 0.518 0.008 -0.510        0.067               5/5   efeito


### 5.14 H7: encoder sem pré-treino ImageNet
Na referência, o encoder ResNet18 começa com os pesos do ImageNet. Aqui ele começa com pesos aleatórios. Todo o resto fica igual, inclusive o teste sintético (não muda a imagem de entrada nem o limiar). Não há fonte consultada sobre o efeito do pré-treino em imagens acústicas (U20). Tempo previsto: cerca de 30 min.

In [19]:
# Treinando a U-Net com o encoder de pesos aleatórios (todo o resto igual à referência)
esc_h7, res_h7 = rodar_lowo(MODELO, SEMENTES, POCOS, imagens, indice_treino, anot, limiares,
                            dict(CFG, pretreino=False), DISP, f"{RES_DIR}/ablacoes/h7_sem_pretreino")
res_h7.to_csv(f"{RES_DIR}/ablacoes/h7_resumos.csv", index=False)

### 5.15 Leitura da H7 e da AUC por patch (H7 e H11)
Aplicando a regra de leitura ao IoU sintético da H7 (o teste é o mesmo da referência). Depois, comparando a AUC por patch no conjunto anotado (71 patches) entre a referência, a H7 e a H11. A AUC não usa o limiar do pseudo-rótulo, mas o conjunto é pequeno: a leitura é qualitativa.

In [20]:
# Aplicando a regra de leitura à H7 (candidata menos referência; o teste sintético é o da referência)
print("H7 sem pré-treino")
print(tabela_ablacao(ref, res_h7, METRICAS_SINT).to_string(index=False))

H7 sem pré-treino
    metrica   ref  cand  delta  dp_sementes pocos_mesmo_sinal              veredito
   iou_sint 0.771 0.758 -0.013        0.030               2/5 sem efeito detectável
  dice_sint 0.824 0.821 -0.003        0.033               2/5 sem efeito detectável
iou_sint_25 0.909 0.875 -0.034        0.019               3/5 sem efeito detectável
iou_sint_75 0.887 0.852 -0.035        0.008               3/5 sem efeito detectável
iou_sint_95 0.518 0.548  0.030        0.080               3/5 sem efeito detectável


In [21]:
# Lendo os escores por patch da referência (um arquivo por fold) e comparando as AUCs da referência, da H7 e da H11
esc_ref = pd.concat([pd.read_csv(f) for f in sorted(glob(f"{RES_DIR}/modelos_sint/unet_resnet18_s*_escores.csv"))],
                    ignore_index=True)
for nome, esc in [("Referência", esc_ref), ("H7 sem pré-treino", esc_h7), ("H11 Multi-Otsu", esc_h11)]:
    por_semente, resumo = avaliar_modelos(esc)
    print(nome, "- AUC por semente:", por_semente.auc.tolist())
    print(resumo.to_string())
    print()

Referência - AUC por semente: [0.783, 0.818, 0.818]
                 auc       f1_lowo        precisao        revocacao       
                mean   std    mean    std     mean    std      mean    std
modelo                                                                    
unet_resnet18  0.806  0.02   0.845  0.041    0.814  0.019     0.879  0.074

H7 sem pré-treino - AUC por semente: [0.82, 0.803, 0.805]
                 auc        f1_lowo        precisao        revocacao       
                mean    std    mean    std     mean    std      mean    std
modelo                                                                     
unet_resnet18  0.809  0.009   0.839  0.044    0.808  0.027     0.873  0.065

H11 Multi-Otsu - AUC por semente: [0.922, 0.924, 0.92]
                 auc        f1_lowo        precisao        revocacao      
                mean    std    mean    std     mean    std      mean   std
modelo                                                                    
unet

### 5.16 Leitura pareada por AUC (regra definida em 18/11/2026)
Há efeito se o intervalo de 95% da diferença de AUC (candidata menos referência, *bootstrap* pareado nos mesmos patches) exclui zero e a diferença média entre sementes passa de 1 desvio-padrão. Caso contrário: "não foi detectada diferença". Vale para a H11 (leitura principal), a H7 (apoio) e, depois, a H9. Com 71 patches, o poder é baixo. Executar de novo a seção 5.2 antes desta, para carregar `ler_auc`.

In [22]:
# Aplicando a regra de leitura por AUC à H7 e à H11 (referência: escores do notebook 04)
for nome, esc in [("H7 sem pré-treino", esc_h7), ("H11 Multi-Otsu", esc_h11)]:
    print(nome)
    print(ler_auc(esc_ref, esc).T.to_string(header=False))
    print()

H7 sem pré-treino
n_patches                                    71
auc_ref                                   0.809
auc_cand                                   0.81
dif_pareada                               0.001
ic95_baixo                               -0.072
ic95_alto                                 0.073
dif_media_sementes                        0.003
dp_sementes                               0.029
veredito            não foi detectada diferença

H11 Multi-Otsu
n_patches               71
auc_ref              0.809
auc_cand             0.923
dif_pareada          0.114
ic95_baixo           0.046
ic95_alto              0.2
dif_media_sementes   0.116
dp_sementes           0.02
veredito            efeito



### 5.17 H11 e H7 contra o Otsu (AUC pareada)
Comparando a referência, a H7 e a H11 com o Otsu nos mesmos 71 patches (escore médio das 3 sementes, IC de 95% por *bootstrap* pareado), como na Fase 2. Serve para dizer se a H11 fecha a distância para o Otsu. Executar de novo a seção 5.2 antes desta.

In [23]:
# Lendo os escores do Otsu (seção 2.12 do notebook 02) e comparando cada configuração com ele
base = pd.read_csv(f"{RES_DIR}/baselines_limiar_escores.csv")
esc_todos = pd.concat([esc_ref.assign(modelo="Referência"), esc_h7.assign(modelo="H7 sem pré-treino"),
                       esc_h11.assign(modelo="H11 Multi-Otsu")], ignore_index=True)
print(comparar_pareado(esc_todos, base, "score_soft", "otsu").to_string(index=False))

           modelo  n  auc_modelo  auc_base  diferenca  ic95_baixo  ic95_alto
   H11 Multi-Otsu 71       0.923     0.932     -0.009      -0.052      0.031
H7 sem pré-treino 71       0.810     0.932     -0.122      -0.203     -0.046
       Referência 71       0.809     0.932     -0.123      -0.213     -0.050


### 5.18 Tabela 1 recalculada com o treino de referência (U25)
A Tabela 1 do artigo veio do treino do notebook 03, e as ablações usam o treino do notebook 04 (`results/modelos_sint/`). As duas execuções diferem na AUC (0,811 contra 0,806 por semente). Esta seção recalcula a AUC, o F1 e a diferença pareada contra o Otsu para a CNN rasa e a U-Net, só com o treino do notebook 04. Definições: **AUC por semente** = AUC dos escores de uma semente, e depois média e desvio entre as 3 sementes; **AUC do escore médio** = AUC da média dos escores das 3 sementes em cada patch (usada nas diferenças pareadas). Executar de novo a seção 5.2 antes desta.

In [24]:
# Lendo os escores dos dois modelos no treino de referência e recalculando a Tabela 1
esc_mod = pd.concat([pd.read_csv(f) for f in sorted(glob(f"{RES_DIR}/modelos_sint/*_s*_escores.csv"))], ignore_index=True)
print(esc_mod.groupby(["modelo", "semente"]).size().unstack())
por_semente, resumo = avaliar_modelos(esc_mod)
print(por_semente.to_string(index=False))
print()
print(resumo.to_string())
print()
base = pd.read_csv(f"{RES_DIR}/baselines_limiar_escores.csv")
print(comparar_pareado(esc_mod, base, "score_soft", "otsu").to_string(index=False))

semente         0   1   2
modelo                   
cnn_rasa       71  71  71
unet_resnet18  71  71  71
       modelo  semente  n   auc  ic95_baixo  ic95_alto  f1_lowo  precisao  revocacao
     cnn_rasa        0 71 0.826       0.698      0.927    0.898     0.841      0.964
     cnn_rasa        1 71 0.780       0.664      0.883    0.804     0.789      0.818
     cnn_rasa        2 71 0.794       0.683      0.893    0.814     0.793      0.836
unet_resnet18        0 71 0.783       0.656      0.892    0.814     0.793      0.836
unet_resnet18        1 71 0.818       0.698      0.917    0.829     0.821      0.836
unet_resnet18        2 71 0.818       0.695      0.919    0.891     0.828      0.964

                 auc        f1_lowo        precisao        revocacao       
                mean    std    mean    std     mean    std      mean    std
modelo                                                                     
cnn_rasa       0.800  0.024   0.839  0.052    0.808  0.029     0.873  0.